# Lesson 2.1 — CloudWatch Metrics, Alarms & Dashboards

**Level: Beginner–Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain CloudWatch namespaces, metrics, dimensions, and statistics
- Query CloudWatch metrics using boto3 and the AWS CLI
- Create CloudWatch Alarms (simple, composite) with SNS notifications
- Build a CloudWatch Dashboard programmatically
- Publish custom metrics from application code

**Exam relevance:** SOA-C03 Domain 1 (22%) — Skills 1.1.1 through 1.1.5

_Reference: [CloudWatch Documentation](https://docs.aws.amazon.com/cloudwatch/latest/monitoring/) | [CloudWatch Metrics Reference](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/aws-services-cloudwatch-metrics.html)_

## 1. CloudWatch Concepts

Think of **CloudWatch** as AWS's central observability service — it collects, stores, and visualizes metrics and logs from virtually every AWS service.

### Core Terminology

| Term | Definition | Example |
|------|-----------|--------|
| **Namespace** | Container for metrics (like a folder) | `AWS/EC2`, `AWS/RDS`, `MyApp/Prod` |
| **Metric** | Time-series data point | `CPUUtilization`, `RequestCount` |
| **Dimension** | Key-value pair to filter a metric | `InstanceId=i-1234567` |
| **Resolution** | Granularity of data | Standard (60s) or High-Resolution (1s) |
| **Statistics** | Aggregation function | `Average`, `Maximum`, `Sum`, `p99` |
| **Period** | Time window for aggregation | 60 seconds, 300 seconds |

### Retention Periods (important for exams!)
| Resolution | Retention |
|-----------|----------|
| < 60 seconds (High-Res) | 3 hours |
| 1 minute | 15 days |
| 5 minutes | 63 days |
| 1 hour | 455 days (15 months) |

### CloudWatch Alarms — States
- **OK** — metric is within the defined threshold
- **ALARM** — metric has breached the threshold
- **INSUFFICIENT_DATA** — not enough data to evaluate (e.g., instance just started)

### Composite Alarms
Combine multiple alarms with AND/OR logic. Reduces alarm noise by only firing when multiple conditions are true simultaneously:
```
ALARM(HighCPU) AND ALARM(HighMemory) → page on-call
ALARM(HighCPU) OR ALARM(HighLatency) → send notification
```

In [ ]:
import boto3
import json
from datetime import datetime, timedelta, timezone

# Initialize CloudWatch client
cw = boto3.client('cloudwatch', region_name='us-east-1')
REGION = 'us-east-1'

print("CloudWatch client initialized.")
print(f"Region: {REGION}")

In [ ]:
# Explore available CloudWatch namespaces in your account
import boto3

cw = boto3.client('cloudwatch', region_name='us-east-1')
paginator = cw.get_paginator('list_metrics')

namespaces = set()
for page in paginator.paginate():
    for metric in page['Metrics']:
        namespaces.add(metric['Namespace'])

print(f"Active CloudWatch Namespaces ({len(namespaces)}):")
for ns in sorted(namespaces):
    print(f"  {ns}")

In [ ]:
# Query CloudWatch metrics — EC2 CPUUtilization example
# If you have no EC2 instances, this will return empty — that's normal

from datetime import datetime, timedelta, timezone
import boto3

cw = boto3.client('cloudwatch', region_name='us-east-1')
ec2 = boto3.client('ec2', region_name='us-east-1')

# Find a running instance to query
instances = ec2.describe_instances(
    Filters=[{'Name': 'instance-state-name', 'Values': ['running']}]
)['Reservations']

if instances:
    instance_id = instances[0]['Instances'][0]['InstanceId']
    print(f"Querying metrics for instance: {instance_id}")
    
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(hours=1)
    
    response = cw.get_metric_statistics(
        Namespace='AWS/EC2',
        MetricName='CPUUtilization',
        Dimensions=[{'Name': 'InstanceId', 'Value': instance_id}],
        StartTime=start_time,
        EndTime=end_time,
        Period=300,           # 5 minute periods
        Statistics=['Average', 'Maximum']
    )
    
    datapoints = sorted(response['Datapoints'], key=lambda x: x['Timestamp'])
    print(f"\nCPU Utilization (last hour, {len(datapoints)} data points):")
    for dp in datapoints[-5:]:  # Show last 5 data points
        print(f"  {dp['Timestamp'].strftime('%H:%M')}  Avg: {dp['Average']:6.2f}%  Max: {dp['Maximum']:6.2f}%")
else:
    print("No running EC2 instances found. Try this query once you launch an instance in a later lesson.")

In [ ]:
# Publish a CUSTOM metric — this is how your applications send telemetry to CloudWatch
import boto3
import random
from datetime import datetime, timezone

cw = boto3.client('cloudwatch', region_name='us-east-1')

# Simulate publishing application-level metrics
# Cost: Custom metrics cost $0.30/month per metric (first 10,000 metrics)
response = cw.put_metric_data(
    Namespace='CloudOpsLearning/Demo',     # Custom namespace
    MetricData=[
        {
            'MetricName': 'OrdersProcessed',
            'Dimensions': [
                {'Name': 'Environment', 'Value': 'dev'},
                {'Name': 'Service', 'Value': 'order-processor'}
            ],
            'Timestamp': datetime.now(timezone.utc),
            'Value': random.randint(50, 200),
            'Unit': 'Count'
        },
        {
            'MetricName': 'OrderProcessingLatencyMs',
            'Dimensions': [
                {'Name': 'Environment', 'Value': 'dev'},
                {'Name': 'Service', 'Value': 'order-processor'}
            ],
            'Timestamp': datetime.now(timezone.utc),
            'Value': random.uniform(50, 500),
            'Unit': 'Milliseconds'
        }
    ]
)

print(f"Published custom metrics. HTTP Status: {response['ResponseMetadata']['HTTPStatusCode']}")
print("Check CloudWatch console → Metrics → CloudOpsLearning/Demo")

In [ ]:
# Create SNS Topic for alarm notifications
import boto3

sns = boto3.client('sns', region_name='us-east-1')

response = sns.create_topic(Name='CloudOpsAlerts')
SNS_TOPIC_ARN = response['TopicArn']
print(f"SNS Topic ARN: {SNS_TOPIC_ARN}")

# In production you would subscribe email/PagerDuty/Slack here:
# sns.subscribe(TopicArn=SNS_TOPIC_ARN, Protocol='email', Endpoint='ops@example.com')

In [ ]:
# Create a CloudWatch Alarm on our custom metric
import boto3

cw = boto3.client('cloudwatch', region_name='us-east-1')
SNS_TOPIC_ARN = sns.create_topic(Name='CloudOpsAlerts')['TopicArn']

# Create alarm: alert if order processing latency exceeds 300ms for 2 consecutive periods
cw.put_metric_alarm(
    AlarmName='HighOrderProcessingLatency',
    AlarmDescription='Alert when order processing latency exceeds 300ms for 2 consecutive 5-minute periods',
    Namespace='CloudOpsLearning/Demo',
    MetricName='OrderProcessingLatencyMs',
    Dimensions=[
        {'Name': 'Environment', 'Value': 'dev'},
        {'Name': 'Service', 'Value': 'order-processor'}
    ],
    Statistic='Average',
    Period=300,              # 5 minutes
    EvaluationPeriods=2,     # Must breach threshold for 2 consecutive periods
    DatapointsToAlarm=2,     # Both data points must breach
    Threshold=300,           # 300ms
    ComparisonOperator='GreaterThanThreshold',
    TreatMissingData='notBreaching',  # Missing data = not an alarm
    AlarmActions=[SNS_TOPIC_ARN],
    OKActions=[SNS_TOPIC_ARN]          # Notify when alarm resolves too
)

print("Alarm 'HighOrderProcessingLatency' created.")

# Verify the alarm
alarm = cw.describe_alarms(AlarmNames=['HighOrderProcessingLatency'])['MetricAlarms'][0]
print(f"State: {alarm['StateValue']}")
print(f"Threshold: {alarm['Threshold']} {alarm['Unit']}")
print(f"Evaluation: {alarm['EvaluationPeriods']} periods x {alarm['Period']}s")

In [ ]:
# Create a CloudWatch Dashboard programmatically
import boto3, json

cw = boto3.client('cloudwatch', region_name='us-east-1')

dashboard_body = {
    "widgets": [
        {
            "type": "text",
            "x": 0, "y": 0, "width": 24, "height": 1,
            "properties": {
                "markdown": "# CloudOps Operations Dashboard\nEnvironment: **dev** | Updated: auto-refresh"
            }
        },
        {
            "type": "metric",
            "x": 0, "y": 1, "width": 12, "height": 6,
            "properties": {
                "title": "Order Processing Latency (ms)",
                "view": "timeSeries",
                "stacked": False,
                "metrics": [
                    ["CloudOpsLearning/Demo", "OrderProcessingLatencyMs",
                     "Environment", "dev", "Service", "order-processor",
                     {"stat": "Average", "label": "Avg Latency"}]
                ],
                "period": 300,
                "region": "us-east-1"
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 1, "width": 12, "height": 6,
            "properties": {
                "title": "Orders Processed (Count)",
                "view": "timeSeries",
                "metrics": [
                    ["CloudOpsLearning/Demo", "OrdersProcessed",
                     "Environment", "dev", "Service", "order-processor",
                     {"stat": "Sum", "label": "Total Orders"}]
                ],
                "period": 300,
                "region": "us-east-1"
            }
        },
        {
            "type": "alarm",
            "x": 0, "y": 7, "width": 24, "height": 3,
            "properties": {
                "title": "Active Alarms",
                "alarms": [f"arn:aws:cloudwatch:us-east-1:{boto3.client('sts').get_caller_identity()['Account']}:alarm:HighOrderProcessingLatency"]
            }
        }
    ]
}

cw.put_dashboard(
    DashboardName='CloudOps-Operations',
    DashboardBody=json.dumps(dashboard_body)
)

print("Dashboard 'CloudOps-Operations' created.")
print(f"View at: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=CloudOps-Operations")

## Exercise

In [ ]:
# TODO: Create a CloudWatch Alarm that:
# - Monitors the 'OrdersProcessed' metric
# - Alarm state when orders DROP BELOW 10 per 5-minute period for 3 consecutive periods
# - (This detects a service outage causing no orders to come through)
# - Sends to the 'CloudOpsAlerts' SNS topic
# - Alarm name: 'LowOrderThroughput'

import boto3
cw = boto3.client('cloudwatch', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')
SNS_TOPIC_ARN = sns.create_topic(Name='CloudOpsAlerts')['TopicArn']

# YOUR CODE HERE
cw.put_metric_alarm(
    AlarmName='LowOrderThroughput',
    # TODO: Fill in the remaining parameters
)

In [ ]:
# Cleanup
import boto3

cw = boto3.client('cloudwatch', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')

# Delete alarms
for alarm_name in ['HighOrderProcessingLatency', 'LowOrderThroughput']:
    try:
        cw.delete_alarms(AlarmNames=[alarm_name])
        print(f"Deleted alarm: {alarm_name}")
    except Exception as e:
        print(f"Alarm {alarm_name}: {e}")

# Delete dashboard
try:
    cw.delete_dashboards(DashboardNames=['CloudOps-Operations'])
    print("Deleted dashboard: CloudOps-Operations")
except Exception as e:
    print(f"Dashboard: {e}")

# Delete SNS topic
try:
    topic_arn = sns.create_topic(Name='CloudOpsAlerts')['TopicArn']
    sns.delete_topic(TopicArn=topic_arn)
    print("Deleted SNS topic: CloudOpsAlerts")
except Exception as e:
    print(f"SNS: {e}")

## Summary

- CloudWatch organizes metrics in **Namespaces** → **Metrics** → **Dimensions**
- **Standard metrics** come from AWS services for free; **custom metrics** cost $0.30/metric/month
- Alarms transition between **OK**, **ALARM**, and **INSUFFICIENT_DATA**
- `EvaluationPeriods` + `DatapointsToAlarm` control how many consecutive threshold breaches trigger an alarm
- **Composite Alarms** reduce alarm noise by combining conditions with AND/OR
- Always send alarm actions to **SNS** for flexible notification routing (email, Slack, PagerDuty)

## Key Exam Points (SOA-C03 Domain 1)
- CloudWatch metric resolution: standard = 60s, high-resolution = 1s
- `TreatMissingData`: `notBreaching` (safe default), `breaching` (aggressive), `ignore`, `missing`
- EC2 detailed monitoring = 1-minute granularity (default is 5-minute)

---

**Next:** [Lesson 2.2 — CloudWatch Logs, Log Insights & CloudWatch Agent](lesson-2.2-cloudwatch-logs.ipynb)